<div style="font-size:11pt; line-height:1.15; text-align:justify; font-family:Times New Roman;">

This project is based on the citizen science platform Planet Hunters TESS, which is part of NASA’s broader effort to involve the public in the search for exoplanets using data from the Transiting Exoplanet Survey Satellite (TESS). The main idea of Planet Hunters TESS is that while automated pipelines are very powerful at scanning large volumes of light curve data, human volunteers can still contribute by visually inspecting light curves and identifying patterns that algorithms may miss or misclassify.

I chose this project because it provides a concrete example of how human pattern recognition and automated signal detection can work together in modern astrophysics. The scientific goal behind the original system is to identify exoplanet transits—small periodic dips in stellar brightness caused by planets passing in front of their host stars—and to distinguish them from noise, stellar variability, or instrumental effects. This is a challenging problem at scale, since TESS produces an extremely large number of light curves, making full manual inspection impossible.

In this project, I developed an interactive Python-based analysis tool that simulates and extends this human–machine collaboration workflow. The purpose of this workflow is not only to detect potential transit signals, but also to understand where automated methods succeed and where they fail, and how human judgment can provide additional validation.

Exoplanet transit detection relies on identifying periodic decreases in stellar brightness caused by a planet crossing the stellar disk, and automated methods such as Box Least Squares (BLS) are widely used because they efficiently search for box‑shaped dips across large data sets. The raw data are discrete samples of stellar flux $f(t_i)$ measured at times $t_i$, and real observations include instrumental noise, cosmic‑ray events, and intrinsic stellar variability that can obscure or mimic transit signals. To mitigate these effects I first remove extreme outliers using sigma clipping, which flags points satisfying $|f(t_i)-\mu|>k\sigma$ with $k=5$ in this implementation so that isolated spikes do not bias subsequent steps. I then estimate a low‑frequency trend $T(t)$ using a median filter and form a normalized light curve $F(t)=f(t)/T(t)$ to isolate short‑duration dips. The BLS algorithm evaluates a grid of trial periods $P$ and durations $\tau$ by folding the light curve modulo each $P$ and fitting a box model $M(\phi)$ where $\phi=\bigl((t-t_0)/P\bigr)\bmod 1$ and the model takes the value $1-\delta$ inside the transit window and $1$ outside; explicitly,
$$
M(\phi)=\begin{cases}
1-\delta, & \text{if } |\phi-\phi_0| < \dfrac{\tau}{2P},\\[6pt]
1, & \text{otherwise},
\end{cases}
$$
and the detection statistic is proportional to the reduction in variance when the box model is applied. Phase folding with the best period aligns candidate transits so that repeated dips appear at the same phase if the detection is physically meaningful, and the phase is computed as
$$
\phi = \left(\frac{t-t_0}{P}\right)\bmod 1.
$$

I used TESS light curves retrieved from the MAST archive and implemented an analysis tool in Python. The program reads the time series and selects the flux column preferring PDCSAP flux when available, then removes non‑finite values and sorts by time. Sigma clipping is applied to the flux series to mask outliers, and a median filter with an odd kernel size is used to estimate the low‑frequency trend; the flattened flux is computed by dividing by the trend and replacing zeros in the trend with a robust median to avoid division artifacts. The BLS search is performed on a dense grid of trial periods $P$ in a chosen range and a set of trial durations $\tau$, and the best period is selected as the one that maximizes the BLS power. Phase folding uses the best period and transit epoch $t_0$ to compute phase $\phi$ and produce a folded light curve for visual inspection. 

Below I present the categories of outcomes observed during analysis.

In cases where both human and algorithm are correct, the raw light curve shows a clear, repeating dip that survives sigma clipping and detrending, the BLS periodogram displays a sharp peak at a single period, and the phase‑folded light curve shows a coherent transit shape with consistent depth and duration. 
<div style="display:flex; gap:15px;">
  <img src="figures/step1_raw.png" width="350">
  <img src="figures/step2_clip.png" width="350">
  <img src="figures/step3_flat.png" width="350">
</div>

<div style="display:flex; gap:10px;">
  <img src="figures/step4_bls.png" width="500">
  <img src="figures/step5_phase.png" width="500">
</div>

<img src="figures/step6_transit.png" width="550">


In cases where the algorithm is correct but the human misses the signal, the transit depth is shallow or the noise level is high so that the dip is not obvious by eye, yet the BLS algorithm finds a period with significant power and the phase‑folded curve reveals a consistent dip when binned or when the transit mask is applied. Insert figures and numerical results for one or more subtle detections here: [INSERT SUBTLE DETECTION FIGURES AND NUMERICAL RESULTS].

In cases where both human and algorithm are wrong, the examples include multi‑periodic signals and pure noise cases that illustrate why both approaches can fail. For multi‑periodic light curves the BLS algorithm, constrained to a single‑period model, often returns a period that maximizes the variance reduction but produces a phase‑folded diagram where the candidate dips are scattered across phases rather than forming a stable transit. For noise‑dominated light curves the algorithm still returns a best period because the box model can fit random fluctuations in a way that reduces variance, but the folded light curve lacks coherent structure. These are the cases where human judgment is most valuable because a human can flag the detection as suspicious even without identifying a correct alternative period. Insert figures and numerical results for representative multi‑periodic and noise‑dominated cases here: [INSERT MULTI‑PERIODIC AND NOISE CASE FIGURES AND NUMERICAL RESULTS].

The mathematical steps of the pipeline clarify why the algorithm can fail and why human judgment is valuable. Sigma clipping removes extreme values but can also remove legitimate short events if thresholds are too strict, and median filter detrending removes low frequency variability but can distort transit shapes when the filter kernel is comparable to the transit duration. The BLS algorithm is optimal for detecting single periodic box‑shaped signals in white noise, but its assumption of a single stable period makes it vulnerable to multi‑periodic systems and to structured noise that mimics periodicity. Because BLS must return a best period, it can be overconfident in cases where the data violate its assumptions. Humans contribute by performing a model validity check: when the phase‑folded points are scattered across phase or when dips vary in depth and timing, a human can flag the detection as suspicious even without identifying a correct alternative period. This human ability to reject spurious algorithmic outputs reduces false positives and focuses follow up on candidates that are both algorithmically significant and visually plausible. The interactive tool supports this workflow by presenting raw curves first, which encourages the human to apply pattern validity checks before being influenced by the algorithmic result.

Overall, humans and algorithms play complementary roles in exoplanet transit detection, with algorithms excelling at finding subtle periodic signals and humans excelling at recognizing when algorithmic assumptions break down. The most scientifically valuable human contribution is not always finding the correct period but rather identifying when a reported period is not physically meaningful, such as when phase‑folded signals are scattered or when the data are dominated by multi‑periodicity or noise.

One of the main contributions of my work is the development of an interactive analysis workflow that explicitly places human judgment into the pipeline rather than treating it as a separate or secondary step. By presenting raw light curves first and then gradually introducing automated analysis outputs, the tool encourages critical evaluation of whether an algorithmic detection is consistent with visual structure in the data. This makes the process closer to how real scientific vetting is done in citizen science projects like Planet Hunters TESS.

From a broader perspective, this project highlights the value of combining computational methods with human intuition. While automated systems are essential for handling the scale of modern astronomical surveys, they are still limited by their assumptions about signal structure. Human participants, even without formal astrophysical training, can contribute by recognizing patterns, questioning uncertain detections, and helping to filter out false positives. In this sense, citizen science does not simply “assist” automation—it actively improves the reliability of the scientific pipeline by introducing an additional layer of interpretation.

</div>

